## Decoder Finetuning Example

In this notebook, you'll get to practice fine-tuning a generative model.

In [18]:
!pip uninstall -y torchao
!pip install torchao>=0.16.0 --no-cache-dir

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
#!pip install transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 21.0 MB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.2 MB/s eta 0:00:00m eta 0:00:010:00:01
  Attempting uninstall: regex
    Found existing installation: regex 2024.11.6
    Uninstalling regex-2024.11.6:
      Successfully uninstalled regex-2024.11.6
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [transformers]m━━━━ 8/9 [transformers]


In [ ]:
#!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 15.1 MB/s eta 0:00:00


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
from transformers import __version__
print(__version__)

5.11.0


For this example, we'll be working with the script of the first episode of Star Trek, the Next Generation.

First, we'll read it in and do some minor cleanup.

In [6]:
with open("script.txt", "r", encoding="utf-8") as f:
    text = f.read()

lines = [line.strip() for line in text.split('\n') if len(line.strip()) > 10]
text = "\n".join(lines)
print(text[:1000])

STAR TREK: THE NEXT GENERATION
"Encounter at Farpoint"
D.C. Fontana
Gene Roddenberry
This script is not for publicaion or reproduction.
No one is authorized to dispose of the same. If los t or
destroyed, please notify the Script Department.
FINAL DRAFT
April 13, 1987
1    EXT. SPACE - STARSHIP (OPTICAL)
The U.S.S. Enterprise NCC 1701-D traveling at warp  speed
through space.
PICARD V.O.
Captain's log, stardate 42353.7.
Our destination is planet Cygnus
IV, beyond which lies the great
unexplored mass of the galaxy.
2    OTHER INTRODUCTORY ANGLES (OPTICAL)
on the gigantic new Enterprise NCC 1701-D.
PICARD V.O.
My orders are to examine Farpoint,
a starbase built there by the
inhabitants of that world.
Meanwhile ...
3    INT. ENGINE ROOM
Huge, with a giant wall diagram showing the immens ity
of this Galaxy Class starship.
PICARD V.O.
(continuing)
... I am becoming better
acquainted with my new command,
this Galaxy Class U.S.S.
Enterprise.
4    CLOSER ON VESSEL DIAGRAM
Showing the details an

First, we need to create our tokenizer.

**Part 1:** Create a tokenizer using "distilgpt2" with the [Autotokenizer.from_pretrained method](https://huggingface.co/docs/transformers/v4.52.3/en/model_doc/auto#transformers.AutoTokenizer).

In [7]:
# Your Code Here
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

During training, all sequences need to be the same lenght, so we need to set a padding token for shorter sequences. We'll use the end of sequence token for this.

In [8]:
tokenizer.pad_token = tokenizer.eos_token

**Part 2:** Use the [encode method](https://huggingface.co/docs/transformers/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.encode) to encode the text. Make sure that this returns PyTorch tensors. Save

In [9]:
# Your Code Here
tokens = tokenizer.encode(text, return_tensors="pt")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (33069 > 1024). Running this sequence through the model will result in indexing errors


In [11]:
torch.save(tokens, "encoded_text.pt")

Now, we'll split the text into shorter chunks.

In [12]:
tokens = tokens[0]

chunk_size = 128
chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens)-chunk_size, chunk_size)]

input_ids = torch.stack([torch.tensor(chunk) for chunk in chunks])

/tmp/ipykernel_1599/3206389813.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.stack([torch.tensor(chunk) for chunk in chunks])


Here's a helper class for our training data.

In [13]:
class ScriptDataset(Dataset):
    def __init__(self, input_ids):
        self.input_ids = input_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": torch.ones_like(self.input_ids[idx]),
            "labels": self.input_ids[idx],
        }

dataset = ScriptDataset(input_ids)

**Part 3:** Make a model named base_model by using the [AutoModelforCausalLM.from_pretrained method](https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoModelForCausalLM), using the pretrained distilgpt2 model.

In [14]:
# Your Code Here

base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
# Match the token_embeddings to the same for the tokenizer.
base_model.resize_token_embeddings(len(tokenizer))

Embedding(50257, 768)

**Part 4:** We'll be finetuning our model using LoRA. Set up a LoraConfig object, lora_config, with rank 8, alpha of 32 and dropout of 0.1. Set the target_modules to ["c_attn"], the bias to "none", and the task_type to TaskType.CAUSAL_LM.

Then, use the [get_peft_model function](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/peft_model#peft.get_peft_model) to create a model using the config object. Save the results to an object named model.

In [16]:
# Your Code Here
lora_config = LoraConfig(r=8, lora_alpha= 32, target_modules= ["c_attn"], bias= "none",lora_dropout= 0.1, task_type = TaskType.CAUSAL_LM)

In [19]:
model = get_peft_model(model =base_model, peft_config=lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


We'll set up a Trainer object and call the train method.

In [20]:
training_args = TrainingArguments(
    output_dir="./xfiles_distilgpt2_lora",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    warmup_steps=5,
    learning_rate=2e-4,
    save_total_limit=1,
    fp16=True,
    report_to="none",

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.711861
20,4.764935
30,4.714433
40,4.736997
50,4.810763
60,4.727884
70,4.685570
80,4.627009
90,4.526954
100,4.539864


TrainOutput(global_step=387, training_loss=4.435578457144803, metrics={'train_runtime': 22.3138, 'train_samples_per_second': 34.687, 'train_steps_per_second': 17.343, 'total_flos': 25368113184768.0, 'train_loss': 4.435578457144803, 'epoch': 3.0})

Before generating new text, let's ensure that we're using GPUs.

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-5): 6 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
             

Here's a helper function to generate new text, given a model.

In [23]:
def generate_text(model, prompt, tokenizer, device, max_new_tokens=100):
    model.eval()    # Ensures that we're generating, not training
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)  # Tokenize the prompt
    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)  # The model generates tokens, so we need to decode those back to words

We'll load back in the base pretrained model for comparison.

In [24]:
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [25]:
def generate_text(model, prompt, tokenizer, device, max_new_tokens=100):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

**Part 5:** Using the generate_text function, try out the prompt "PICARD" with both the pretrained distilgpt2 model (base_model), and the finetuned model. Try other prompts, too.

In [42]:
# Your Code Here

prompt = "DATA SCIENCE"

print(generate_text(base_model, prompt, tokenizer, device,max_new_tokens= 100))

DATA SCIENCE – The idea behind this project has been a lot of fun!


The project was conceived as a demonstration for the development of the novel series. Although the project is somewhat out of the picture, the series is very clear in this video where it demonstrates the concept of the game.
SCIENCE – The idea behind this project has been a lot of fun! SCIENCE – The idea behind this project has been a lot of fun!
The idea is about the development of the novel series


In [44]:
prompt = "DATA SCIENCE"
print(generate_text(model, prompt, tokenizer, device, max_new_tokens=100))

DATA SCIENCE-POINT
TOMOR (voice)
...to be here.
"Do you understand?"
"I don't understand you."
I'm ready to tell her.
I have a moment to apologize to.
I take her to the bridge. She's gone there!
"Oiled?"
"I can't help but feel that I'm a little bit wounded by the sight of my
self being forced to stand
to face down again...
"I


**Bonus:** See what happens if you allow more training epochs. You've also been provided all of the scripts from season 1. How does the model do when given more examples?